# 4.05 Arboles Azarosos — Submit a Kaggle del Top-10 de `z422`

Toma el **Top `PARAM$top_n`** de combinaciones de `z422` (`gridsearch_local.txt`) y para cada una entrena el ensemble completo sobre el **100% de 202107** (igual que `z420`), subiendo un submit a Kaggle en cada punto de `PARAM$grabar` (`1, 2, 4, 8, 16, 32` arboles). Es la misma logica de `z420`/`z421`, pero en vez de barrer una grilla nueva o correr una sola combinacion, repite el ensemble completo para cada una de las combinaciones ya identificadas como las mejores en la validacion local.

#### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

---

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 22 01:07:31 AM 2026"

In [1]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,668153,35.7,1473300,78.7,1425957,76.2
Vcells,1236310,9.5,8388608,64.0,1978712,15.1


In [2]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia, y cuantas combinaciones del ranking de `z422` subir a Kaggle

In [3]:
PARAM <- list()
PARAM$semilla_primigenia <- 346321

PARAM$rpart$cp <- -1 # fijo, igual que en z422/z423

# voy a generar PARAM$num_trees_max arboles por combinacion, igual que z420/z421
PARAM$num_trees_max <- 32

# puntos del ensemble donde se sube un submit a Kaggle, igual que en z420/z421
PARAM$grabar <- c(1, 2, 4, 8, 16, 32)

PARAM$top_n <- 10 # cuantas de las mejores combinaciones de z422 se suben a Kaggle

# donde esta el ranking de z422 del que se toma el Top-N
PARAM$archivo_grid_origen <- "/content/buckets/b1/exp/exp430/gridsearch_localNuevasCombinacionesCorrida2.txt"

In [4]:
PARAM

$semilla_primigenia
[1] 346321

$rpart
$rpart$cp
[1] -1


$num_trees_max
[1] 32

$grabar
[1]  1  2  4  8 16 32

$top_n
[1] 10

$archivo_grid_origen
[1] "/content/buckets/b1/exp/exp430/gridsearch_localNuevasCombinacionesCorrida2.txt"

In [5]:
# chequeo cuantos submits va a consumir esto, contra el limite diario de 100
qty_submits_planificados <- PARAM$top_n * length(PARAM$grabar)

message("combinaciones a subir: ", PARAM$top_n)
message("submits planificados: ", qty_submits_planificados, " / 100 por dia")

if (qty_submits_planificados > 100) {
  warning("Esto planifica mas submits que el limite diario, achica PARAM$top_n o PARAM$grabar")
}

combinaciones a subir: 10

submits planificados: 60 / 100 por dia



In [8]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "experimento_430"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [6]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# defino los dataset de entrenamiento y aplicacion, igual que z420: 100% de 202107
#  para entrenar, 202109 (sin clase) es donde se aplica y se sube a Kaggle
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

### Top-N combinaciones del ranking de `z422`

In [7]:
tb_grid_origen <- fread(PARAM$archivo_grid_origen)

# ranking por ganancia del ensemble completo (ultimo punto de grabar EN z422,
#  no confundir con PARAM$grabar de este script que llega hasta 32)
tb_top <- tb_grid_origen[arbolito == max(arbolito)]
setorder(tb_top, -ganancia)
tb_top <- tb_top[1:PARAM$top_n]
#tb_top <- tb_top[c(1,3,5,8,10)]
tb_top

combo_id,feature_fraction,cp,minsplit,minbucket,maxdepth,arbolito,ganancia
<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>
67,0.6,-1,1000,50,12,32,501583333
68,0.6,-1,1000,50,14,32,501000000
10,0.5,-1,1000,100,10,32,499166667
11,0.5,-1,1000,100,12,32,498416667
16,0.5,-1,1000,150,14,32,497666667
15,0.5,-1,1000,150,12,32,497500000
65,0.6,-1,1000,50,8,32,496833333
75,0.6,-1,1000,100,12,32,496750000
76,0.6,-1,1000,100,14,32,496666667


### Submit a Kaggle: Top-N combinaciones, ensemble completo cada una

In [10]:
# archivo donde se guarda el checkpoint (que combinacion+arbolito ya se subio)
archivo_grid <- "gridsearch_kaggleNuevasCombCorridas2.txt"

if (file.exists(archivo_grid)) {
  tb_grid <- fread(archivo_grid)
} else {
  tb_grid <- data.table(
    combo_id = integer(),
    feature_fraction = numeric(),
    cp = numeric(),
    minsplit = integer(),
    minbucket = integer(),
    maxdepth = integer(),
    arbolito = integer(),
    archivo_kaggle = character()
  )
}

for (i in seq_len(nrow(tb_top))) {

  v_combo_id         <- tb_top[i, combo_id]
  v_feature_fraction <- tb_top[i, feature_fraction]
  v_minsplit         <- tb_top[i, minsplit]
  v_minbucket        <- tb_top[i, minbucket]
  v_maxdepth         <- tb_top[i, maxdepth]

  # si esta combinacion ya tiene TODOS los puntos de grabar subidos, la salteo entera
  puntos_hechos <- tb_grid[combo_id == v_combo_id, arbolito]
  if (all(PARAM$grabar %in% puntos_hechos)) next

  message("combo ", v_combo_id, " : feature_fraction=", v_feature_fraction,
    " minsplit=", v_minsplit, " minbucket=", v_minbucket, " maxdepth=", v_maxdepth)

  rpart_control <- list(
    cp= PARAM$rpart$cp,
    minsplit= v_minsplit,
    minbucket= v_minbucket,
    maxdepth= v_maxdepth
  )

  # misma semilla para todas las combinaciones, asi la unica diferencia entre
  #  combinaciones es el hiperparametro, no el azar (igual que z421)
  set.seed(PARAM$semilla_primigenia)

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob_acumulada := 0]

  # entreno el ensemble arbol por arbol, subiendo un submit en cada punto de PARAM$grabar
  for (arbolito in seq(PARAM$num_trees_max)) {
    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * v_feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    campos_random <- paste(campos_random, collapse= " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    modelo <- rpart(formulita, data= dtrain, xval= 0, control= rpart_control)
    prediccion <- predict(modelo, dfuture, type= "prob")
    tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

    if (!(arbolito %in% PARAM$grabar)) next

    # si este punto puntual ya fue subido en un intento anterior, no lo repito
    #  (igual tuve que re-entrenar los arboles anteriores para llegar hasta aca)
    if (arbolito %in% puntos_hechos) next

    umbral_corte <- arbolito / 40
    tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

    archivo_kaggle <- paste0("KA430_combo_", sprintf("%.3d", v_combo_id),
      "_arb", sprintf("%.3d", arbolito), ".csv")
    fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
      file= archivo_kaggle, sep= "," )

    # subida a Kaggle, el mensaje lleva la combinacion y el arbolito para poder identificarlos despues
    comando <- "kaggle competitions submit"
    competencia <- "-c utn-2026-inicial"
    arch <- paste( "-f", archivo_kaggle)
    mensaje <- paste0("-m 'top10 combo=", v_combo_id, " ff=", v_feature_fraction,
      " cp=", PARAM$rpart$cp, " minsplit=", v_minsplit, " minbucket=", v_minbucket,
      " maxdepth=", v_maxdepth, " arbolito=", arbolito, "'")
    linea <- paste( comando, competencia, arch, mensaje)
    salida <- system(linea, intern=TRUE)
    cat(salida)

    tb_grid <- rbindlist(list( tb_grid, data.table(
      combo_id= v_combo_id,
      feature_fraction= v_feature_fraction,
      cp= PARAM$rpart$cp,
      minsplit= v_minsplit,
      minbucket= v_minbucket,
      maxdepth= v_maxdepth,
      arbolito= arbolito,
      archivo_kaggle= archivo_kaggle
    )))

    # grabo el checkpoint despues de cada submit, para poder retomar
    #  sin volver a gastar submits ya usados
    fwrite(tb_grid, file= archivo_grid, sep= "\t")
  }
}

combo 67 : feature_fraction=0.6 minsplit=1000 minbucket=50 maxdepth=12



combo 68 : feature_fraction=0.6 minsplit=1000 minbucket=50 maxdepth=14



combo 10 : feature_fraction=0.5 minsplit=1000 minbucket=100 maxdepth=10



combo 11 : feature_fraction=0.5 minsplit=1000 minbucket=100 maxdepth=12



combo 16 : feature_fraction=0.5 minsplit=1000 minbucket=150 maxdepth=14



combo 15 : feature_fraction=0.5 minsplit=1000 minbucket=150 maxdepth=12



combo 65 : feature_fraction=0.6 minsplit=1000 minbucket=50 maxdepth=8



combo 75 : feature_fraction=0.6 minsplit=1000 minbucket=100 maxdepth=12



combo 76 : feature_fraction=0.6 minsplit=1000 minbucket=100 maxdepth=14



combo 71 : feature_fraction=0.6 minsplit=1000 minbucket=75 maxdepth=12



In [ ]:
format(Sys.time(), "%a %b %d %X %Y")